In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import json

In [34]:
# 1. Load the Dataset
df = pd.read_csv('sign_data.csv')

# --- NEW: FILTER FOR SPECIFIC LETTERS ---
target_letters = ['A', 'D', 'E', 'O', 'R', 'Y']
df = df[df['Sign'].isin(target_letters)]
# ----------------------------------------

# Features are the 'Distance_0' through 'Distance_209' columns
X = df.drop('Sign', axis=1).values
y = df['Sign'].values

In [35]:
# 2. Preprocess Labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(np.unique(y_encoded))

# Save label names so your script knows the new order
label_names = list(label_encoder.classes_)
with open('labels.json', 'w') as f:
    json.dump(label_names, f)

# --- NEW: map each letter to the gesture name you want displayed ---
gesture_display = {
    'A': 'Fist',
    'B': 'Open hand',
    'D': 'Point',
    'O': 'Circle',
    'R': 'Crosses',
    'Y': 'Call'
}
with open('gesture_display.json', 'w') as f:
    json.dump(gesture_display, f)
# ----------------------------------------

print(f"Dataset filtered! Training on classes: {label_names}")

Dataset filtered! Training on classes: ['A', 'D', 'E', 'O', 'R', 'Y']


In [36]:
# 3. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [37]:
# 4. Build a Stronger Model (Deep Neural Network)
model = models.Sequential([
    layers.Input(shape=(210,)), # 210 distance features
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [38]:
# 5. Train
print(f"Training on {num_classes} gestures...")
history = model.fit(
    X_train, y_train,
    epochs=60,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Training on 6 gestures...
Epoch 1/60
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.8542 - loss: 0.5768 - val_accuracy: 0.5083 - val_loss: 1.5031
Epoch 2/60
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0285 - val_accuracy: 0.5083 - val_loss: 1.3457
Epoch 3/60
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0084 - val_accuracy: 0.6917 - val_loss: 1.2159
Epoch 4/60
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0025 - val_accuracy: 0.9417 - val_loss: 1.1121
Epoch 5/60
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9979 - loss: 0.0052 - val_accuracy: 1.0000 - val_loss: 1.0247
Epoch 6/60
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0029 - val_accuracy: 1.0000 - val_loss: 0.9538
Epoch 7/60
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0020 - val_accuracy: 1.0000 - val_loss: 0.8632
Epoch 8/60
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0021 

In [39]:
# 6. Save the Model
model.save('asl_gesture_model.h5')
print("Success! Model saved as asl_gesture_model.h5")

Success! Model saved as asl_gesture_model.h5
